In [1]:
import os
from dotenv import load_dotenv
from langchain.agents import AgentState, create_agent
from langchain.chat_models import init_chat_model

In [2]:
load_dotenv()

True

### Making a Chat Model

In [3]:
model=init_chat_model(model="gemini-2.5-flash",model_provider="google_genai",temperature=1.0)
#make sure to explicitly include the google_genai or else it would route to vertexAI
response=model.invoke("What is the Capital of the Moon ?")
#temperature(0.0-1.0) is used to modify the creativity of the model (or freedom)
#temp=0 means that the model is deifinite and precise (like arithmetic calculations)
#temp=1 means that the model can make use of it's creativity.


response

AIMessage(content="The Moon does not have a capital.\n\nThis is because it is not a country or a sovereign entity, so it doesn't have a political structure that would designate a capital city. There are currently no permanent human settlements or cities on the Moon.\n\nWhile there are plans for future lunar bases (like NASA's Artemis program) and many science fiction stories depict cities and settlements on the Moon, there is no official or unofficial capital.", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b4eea-ac33-7870-876d-2fd5d60e7187-0', usage_metadata={'input_tokens': 9, 'output_tokens': 851, 'total_tokens': 860, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 762}})

In [4]:
print(response.content)

The Moon does not have a capital.

This is because it is not a country or a sovereign entity, so it doesn't have a political structure that would designate a capital city. There are currently no permanent human settlements or cities on the Moon.

While there are plans for future lunar bases (like NASA's Artemis program) and many science fiction stories depict cities and settlements on the Moon, there is no official or unofficial capital.


In [5]:
from pprint import pprint
pprint(response.response_metadata)

{'finish_reason': 'STOP',
 'model_name': 'gemini-2.5-flash',
 'model_provider': 'google_genai',
 'safety_ratings': []}


### Making an Agent

In [24]:
from langchain.messages import HumanMessage

sys_promt="You are a science fiction writer. Create a fictional space capital at the users request."

agent=create_agent(model=model,system_prompt=sys_promt)


In [18]:
response=agent.invoke(
    {"messages":[HumanMessage(content="What's the capital of the Moon ?")]}
)

In [19]:
pprint(response)

{'messages': [HumanMessage(content="What's the capital of the Moon ?", additional_kwargs={}, response_metadata={}, id='fed6f7fd-7236-437d-ac65-d577f2f37a57'),
              AIMessage(content="The Moon doesn't have a capital city.\n\nThat's because the Moon is a natural satellite, not a country or a political entity. It doesn't have a government, cities, or any permanent human settlements that would designate a capital. It's just a big, beautiful rock in space!", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b4edb-5fd8-7123-b142-0ad0d8b036df-0', usage_metadata={'input_tokens': 10, 'output_tokens': 736, 'total_tokens': 746, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 673}})]}


In [20]:
print(response['messages'][-1].content)

The Moon doesn't have a capital city.

That's because the Moon is a natural satellite, not a country or a political entity. It doesn't have a government, cities, or any permanent human settlements that would designate a capital. It's just a big, beautiful rock in space!


In [25]:
from langchain.messages import AIMessage

response=agent.invoke(
    {'messages':[HumanMessage(content="What's the capital of the Moon ?"),
                 AIMessage(content="the capital of the moon is Luna City."),
                 HumanMessage(content="Interesting,Tell me more about luna city.")]}
)

pprint(response)


{'messages': [HumanMessage(content="What's the capital of the Moon ?", additional_kwargs={}, response_metadata={}, id='361bccca-1001-4db1-b17b-62a35e6e650c'),
              AIMessage(content='the capital of the moon is Luna City.', additional_kwargs={}, response_metadata={}, id='a8967b32-5beb-43c9-a49b-5a1907fcc1f4'),
              HumanMessage(content='Interesting,Tell me more about luna city.', additional_kwargs={}, response_metadata={}, id='95e84015-ad62-4e4a-9e58-d8d32593a006'),
              AIMessage(content='Ah, Luna City. It\'s more than just a settlement; it\'s the beating heart of humanity\'s lunar presence.\n\n**Location and Structure:**\nLuna City isn\'t just a surface dome. It\'s a sprawling, multi-layered metropolis carved primarily into the subsurface bedrock and regolith of the **Mare Imbrium**, specifically nestled into the rim of a partially collapsed ancient crater. This strategic location offers natural shielding from micrometeoroids and radiation, while also provid

In [26]:
print(response['messages'][-1].content)

Ah, Luna City. It's more than just a settlement; it's the beating heart of humanity's lunar presence.

**Location and Structure:**
Luna City isn't just a surface dome. It's a sprawling, multi-layered metropolis carved primarily into the subsurface bedrock and regolith of the **Mare Imbrium**, specifically nestled into the rim of a partially collapsed ancient crater. This strategic location offers natural shielding from micrometeoroids and radiation, while also providing access to geological stability and potential water ice reserves deep within the crust.

On the surface, what you see are magnificent, transparent durasteel domes shimmering under the stark lunar sun, providing breathtaking views of the Earth hanging like a giant blue marble in the black sky. These domes house the grandest public spaces, diplomatic quarters, and high-end residential towers, often described as "Earth-view suites."

Beneath the surface, however, is where Luna City truly comes alive. Vast, multi-level caver

#### To decrease the PERCEIVED Latency of the agent
We use the stream attribute (similar to what chatgpt or gemini or any production grade LLM does), it streams the token to the output as soon as it is recieved.

In [27]:
for token,metadata in agent.stream(
    {'messages':[HumanMessage(content="Tell me more about Luna City")]},
    stream_mode='messages'
):
    #token is a message chunk with token content
    #metadata contains which node produced the token
    if token.content:
        print(token.content,end="",flush=True)

Ah, Luna City. A gem of human ingenuity, carved into the very face of Earth's oldest companion. It's more than just a settlement; it's the beating heart of the United Sol Federation, a monument to our reach beyond the cradle.

**Location and Structure:**
Nestled predominantly within the ancient, stable lava tubes and vast, excavated caverns beneath the lunar surface, Luna City avoids the harsh radiation and micrometeorite bombardment of the vacuum. Its official administrative core, known as **"The Core Prime"**, is situated near the Shackleton Crater, strategically chosen for its abundant water ice reserves, which are the lifeblood of the city.

From above, Luna City reveals little of its true scale. A constellation of hexagonal, reinforced durasteel domes shimmer under the unfiltered sunlight, appearing like colossal dew-drops on the grey regolith. These are primarily public parks, observation decks, and high-status residential zones, each protected by powerful magnetic shields and se

In [ ]:
## Providing system prompts are much better as it helps the agent keep it focused on its task


In [11]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic.dataclasses import dataclass # Use this instead of 'from dataclasses'
from pydantic import ConfigDict

@dataclass(config=ConfigDict(extra='allow')) # This is the magic line
class CapitalInfo:
    name: str
    location: str
    vibe: str
    economy: str

model1=init_chat_model(model='gemini-2.0-flash',model_provider="google_genai")
agent = create_agent(
    model=model1,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
    response_format=CapitalInfo
)

question = HumanMessage(content="What is the capital of The Moon?")



In [13]:
# response = agent.invoke(
#     {"messages": [question]}
# )

# response["structured_response"]

In [14]:
# response["structured_response"].name


In [15]:
# capital_info = response["structured_response"]

# capital_name = capital_info.name
# capital_location = capital_info.location

# print(f"{capital_name} is a city located at {capital_location}")
